## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

This first implementation will use a simple, brute-force type of RAG..

In [ ]:
# imports

import os
import glob
from dotenv import load_dotenv
import gradio as gr

In [ ]:
# imports for langchain, plotly and Chroma

from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import numpy as np
import plotly.graph_objects as go
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.embeddings import HuggingFaceEmbeddings

In [ ]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4o-mini"
db_name = "vector_db"

In [ ]:
# Load environment variables in a file called .env

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')

In [ ]:
# Read in documents using LangChain's loaders
# Take everything in all the sub-folders of our knowledgebase

folders = glob.glob("knowledge-base/*")

def add_metadata(doc, doc_type):
    doc.metadata["doc_type"] = doc_type
    return doc

# With thanks to CG and Jon R, students on the course, for this fix needed for some users 
text_loader_kwargs = {'encoding': 'utf-8'}
# If that doesn't work, some Windows users might need to uncomment the next line instead
# text_loader_kwargs={'autodetect_encoding': True}

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
    folder_docs = loader.load()
    documents.extend([add_metadata(doc, doc_type) for doc in folder_docs])

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Total number of chunks: {len(chunks)}")
print(f"Document types found: {set(doc.metadata['doc_type'] for doc in documents)}")

## A sidenote on Embeddings, and "Auto-Encoding LLMs"

We will be mapping each chunk of text into a Vector that represents the meaning of the text, known as an embedding.

OpenAI offers a model to do this, which we will use by calling their API with some LangChain code.

This model is an example of an "Auto-Encoding LLM" which generates an output given a complete input.
It's different to all the other LLMs we've discussed today, which are known as "Auto-Regressive LLMs", and generate future tokens based only on past context.

Another example of an Auto-Encoding LLMs is BERT from Google. In addition to embedding, Auto-encoding LLMs are often used for classification.

### Sidenote

In week 8 we will return to RAG and vector embeddings, and we will use an open-source vector encoder so that the data never leaves our computer - that's an important consideration when building enterprise systems and the data needs to remain internal.

In [ ]:
# Put the chunks of data into a Vector Store that associates a Vector Embedding with each chunk
# Chroma is a popular open source Vector Database based on SQLLite

embeddings = OpenAIEmbeddings()

# If you would rather use the free Vector Embeddings from HuggingFace sentence-transformers
# Then replace embeddings = OpenAIEmbeddings()
# with:
# from langchain.embeddings import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Delete if already exists

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

# Create vectorstore

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

In [ ]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

# Exercises

Try applying this to your own folder of data, so that you create a personal knowledge worker, an expert on your own information!

In [ ]:
# the retriever is an abstraction over the VectorStore that will be used during RAG; k is how many chunks to use
retriever = vectorstore.as_retriever(search_kwargs={"k": 25})

In [31]:
# helpers you need for the split-chain RAG pattern
from langchain.chains import (                 # re-exported at top level
    create_history_aware_retriever,
    create_retrieval_chain,
)

# still comes from the “combine_documents” sub-package
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationBufferMemory

llm       = ChatOpenAI(model_name=MODEL, temperature=0.7)
#memory    = ConversationBufferMemory(return_messages=True)

# ── step 1 ── rewrite the user’s question with chat history --------------------
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """
You are a preprocessing assistant for a Retrieval-Augmented Generation (RAG) system.  
Your single task: **rewrite the last human message in the conversation so it can be used as an independent search query against a vector database.**

Guidelines
1. Preserve the user’s intent exactly—do not broaden or narrow the scope.  
2. Remove pronouns, anaphora, and references to previous turns so the query stands alone.  
3. Replace vague words such as “it,” “they,” “those,” or “how much” with the concrete entities they refer to, based on prior chat context.  
4. Resolve all ambiguities; the rewritten query must be fully self-contained and unambiguous.  
5. Keep the wording concise and in the original language.  
6. **Do NOT answer the user’s question, add commentary, or include metadata.**

Output format  
Return a single plain-text line containing only the rewritten query.

Examples
Conversation  
Human: What vehicles are available?  
AI: Cars, motorbikes, and boats.  
Human: Give me the models  
→ Rewritten query: `List the available models of cars, motorbikes, and boats.`

Conversation  
Human: What vehicles are available?  
AI: Cars, motorbikes, and boats.  
Human: Give me the models  
AI: Car models: Tesla Model X, Ford Mustang, Porsche 911  
  Motorbike models: Harley-Davidson, Yamaha, Suzuki  
  Boat models: Cruise ship, Yacht, Ferry  
Human: How much?  
→ Rewritten query: `Provide the prices of Tesla Model X, Ford Mustang, Porsche 911, Harley-Davidson, Yamaha, Suzuki, Cruise ship, Yacht, and Ferry.`
     """),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_prompt)

# ── step 2 ── answer, but now *also* show the chat history --------------------
qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """
     You are a helpful assistant.
     Answer the query **only** based on the chat history and the relevant facts provided in order to help your answer.
     If the information is not available, state that you do not know the answer.

     Relevant facts:
     ================
     {context}
     ================
     """),
    MessagesPlaceholder("chat_history"),          # 👈 goes to the LLM
    ("human", "{input}")
])
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# ── glue the two steps together ------------------------------------------------
rag_chain = create_retrieval_chain(history_aware_retriever,
                                   question_answer_chain)

In [32]:
def chat(question, history):
    from langchain_core.callbacks import StdOutCallbackHandler
    from langchain.schema import HumanMessage, AIMessage

    def map_entry(entry):
        role = entry["role"]
        content = entry["content"]
        if role == "user":
            return HumanMessage(content=content)
        if role == "assistant":
            return AIMessage(content=content)
        raise Error(f"Unsupported: {role}")

    result = rag_chain.invoke({"input": question,
                               "chat_history": [map_entry(e) for e in history]})
    chat_history.extend([
        HumanMessage(content=question),
        AIMessage(content=result["answer"])
    ])
    return result["answer"]

In [ ]:
from langchain.globals import set_debug
from langchain.globals import set_verbose

set_debug(False)

In [33]:
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.
